# Objective

Establish the accuracy of a base LLM (Mistral) on dialogue summarization


# Setup

In [1]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.5/203.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/

Found existing installation: unsloth 2025.4.3
Uninstalling unsloth-2025.4.3:
  Successfully uninstalled unsloth-2025.4.3
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-_mb20riv
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-_mb20riv
  Resolved https://github.com/unslothai/unsloth.git to commit 2a65066e5ae2b75fd20ec5184dd731c284bcfcc4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.4.3-py3-none-any.whl size=205495 sha256=68bdeba397f6ae205d455db80b054263b266d33301b5c003ec735ab234d1902d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ttus8ysi/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth


In [1]:
#download datasets evaluate rouge_score and bert score
!pip install -q datasets==3.0.1 evaluate==0.4.3 bert_score

In [2]:
# !pip install triton==3.1.0

# Imports

In [3]:
import torch
import evaluate

from datasets import load_dataset
from tqdm import tqdm

from unsloth import FastLanguageModel

<ipython-input-3-d0f8a4c73c62>:7: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Test Data

Source: https://huggingface.co/datasets/knkarthick/dialogsum

In [4]:
dataset = load_dataset("knkarthick/dialogsum")

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [5]:
test_data_size = 30

In [6]:
test_dataset = dataset['test'].shuffle(seed=42).select(range(test_data_size))

In [7]:
test_dialogues = [sample['dialogue'] for sample in test_dataset]
test_summaries = [sample['summary'] for sample in test_dataset]

# Model

In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True
)

==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

In [9]:
FastLanguageModel.for_inference(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): Mis

In [10]:
type(model)

transformers.models.mistral.modeling_mistral.MistralForCausalLM

# Inference

In [11]:
summarization_prompt_template = """[INST]
Summarize the dialogue mentioned in the user input. Be specific and concise in your summary.
Ensure that you retain the entities mentioned in the dialogue in your summary.

### User Input:
{dialogue}
[/INST]
"""

In [12]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): Mis

In [13]:
predicted_summaries = []

In [14]:
for gold_dialogue in tqdm(test_dialogues):

    try:
        prompt = summarization_prompt_template.format(dialogue=gold_dialogue)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=0,
            pad_token_id=tokenizer.eos_token_id
        )

        prediction = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[-1]:],
            skip_special_tokens=True,
            cleanup_tokenization_spaces=True
        )
        print(prediction)
        predicted_summaries.append(prediction)

    except Exception as e:
        print(e) # log error and continue
        continue

  3%|▎         | 1/30 [00:11<05:20, 11.05s/it]

Person1 and Person2 engage in a conversation about various pilgrimages in different religions. Person2 is watching a program about the Islamic pilgrimage to Mecca, called 'haj. They discuss the significance of Mecca as the spiritual center of Islam and the belief that able Muslims should make this pilgrimage at least once in their lifetime. Person1 asks about the large number of pilgrims and the potential dangers, to which Person2 explains the challenges of managing such large crowds and the efforts made by the Saudi government to limit the number of pilgrims. They also discuss other pilgrimages in Christianity,


  7%|▋         | 2/30 [00:15<03:24,  7.29s/it]

Person1 requests to send a package using first-class mail with insurance for $50. Person2 informs Person1 that they need to get stamps, including a book of 22 and three airmail, at the stamp window next to general delivery. Person1 inquires if they can also get money orders there, but is told they need to go to the window three windows down the hall for that.


 10%|█         | 3/30 [00:19<02:33,  5.69s/it]

...

Person2 worked as a secretary to the General Manager right after graduating from college in 1998. After two years, they were promoted to Personnel Manager, handling all personnel matters. Although they enjoyed working with the people, the job was poorly paid. After about a year, they left to pursue a position in the Sales Department, where they currently work.


 97%|█████████▋| 29/30 [02:40<00:05,  5.11s/it]

Person1 asks Person2 for their opinion on his new suit. Person2 thinks it's not bad but is reminded of a similar suit they saw at a department store last week. Person1 reveals he didn't buy it there and instead purchased it in a big shopping center for $150. Person2 expresses their belief that it's not a good bargain.


100%|██████████| 30/30 [02:48<00:00,  5.61s/it]

Person1 asks Person2 about the number of people in their party, to which Person2 replies there are three: two adults and one kid. Person1 confirms it's for the buffet. Person2 inquires about the cost and Person1 informs them that it's 30 yuan per adult and 20 yuan per kid. Person2 asks about the location of the food and Person1 directs them to the tables with cold dishes and vegetables, while the hot dishes are on the other side. Person2 asks if there are extra charges for drinks and Person1 states that there'


# Evaluation

In [15]:
predicted_summaries[1]

'Person1 requests to send a package using first-class mail with insurance for $50. Person2 informs Person1 that they need to get stamps, including a book of 22 and three airmail, at the stamp window next to general delivery. Person1 inquires if they can also get money orders there, but is told they need to go to the window three windows down the hall for that.'

In [16]:
bert_scorer = evaluate.load("bertscore")

In [17]:
score = bert_scorer.compute(
    predictions=predicted_summaries,
    references=test_summaries,
    lang='en',
    rescale_with_baseline=True
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
sum(score['f1'])/len(score['f1'])

0.18733006864786148